# Ingestion de données via une requête API de sales motion

## Import des librairies


In [ ]:
import os

import pandas as pd
import pandas_gbq
import requests
from dotenv import load_dotenv

## Définition des fonctions utiliser

In [44]:
### une liste d'entreprise utiliser pour notre exemple de requête

companies = ["stripe.com", "databricks.com", "qonto.com", "blablacar.com"]

# définition d'une fonction pour récupérer l'ID des entreprises sur salesmotion via le nom de domaine


def get_company_account_id(company: str):

    url = f"https://api.salesmotion.io/v1/companies/by-input/{company}"
    headers = {"Authorization": f"Bearer {api_key}"}
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Erreur {response.status_code} : {response.text}")
        return None


# définition d'une 2eme fonction pour récupérer les offres d'emplois d'une entreprise à partir de son ID


def get_company_jobs(account_id: str):
    url = f"https://api.salesmotion.io/v1/companies/{account_id}/job-openings"
    header = {"Authorization": f"Bearer {api_key}"}
    response = requests.get(url, headers=header)

    if response.status_code == 200:
        return response.json()
    else:
        print(
            f"Erreur {response.status_code} pour l'account {account_id}: {response.text}"
        )
        return None

## Boucle sur la liste d'entreprise pour récupérer leurs offres d'emploi

In [ ]:
# boucle pour récupérer les offres d'emplois pour chaque entreprise de la liste via les fonctions précédentes

all_jobs = []
job_counter = 0

for company in companies:
    account_id = get_company_account_id(company)["data"][
        "id"
    ]  # on récupère l'ID de l'entreprise

    if account_id:
        jobs = get_company_jobs(account_id)["data"]  # on récupère la liste des jobs
        for job in jobs:
            job_counter += 1
            job["job_number"] = job_counter  # on ajoute un compteur pour chaque job
            job["_ingested_at"] = pd.Timestamp.now(
                tz="UTC"
            ).isoformat()  # on ajoute la date d'ingestion
            all_jobs.append(job)  # on ajoute le job à la liste globale

## Enregistrement en DataFrame & expert vers bigquery 

In [58]:
### transformation de la liste de jobs en DataFrame et export vers bigquery

df = pd.DataFrame(all_jobs)

df.head(3)

,id,companyId,status,url,title,companyName,location,postedOn,seniorityLevel,employmentType,jobFunction,industries,job_number,company_domain,_ingested_at,applicants
0,O0o0QrK5sQhoXgAnii0Hr,p9Jh396TLE7TLiPb2KXq,online,https://ca.linkedin.com/jobs/view/data-scienti...,"Data Scientist, Link",Stripe,"Toronto, Ontario, Canada",2026-08-24T00:00:00,Mid-Senior level,Full-time,Engineering and Information Technology,"Software Development, Financial Services, and ...",1,stripe.com,2026-08-25T09:38:48.397812+00:00,NaN
1,uHstOR3YwfLgSX0IlIify,p9Jh396TLE7TLiPb2KXq,online,https://www.linkedin.com/jobs/view/staff-softw...,"Staff Software Engineer, Startup Products",Stripe,United States,2026-08-22T00:00:00,Mid-Senior level,Full-time,Engineering and Information Technology,"Software Development, Financial Services, and ...",2,stripe.com,2026-08-25T09:38:48.398084+00:00,NaN
2,e3SVmsglHPHf5AmCiURFg,p9Jh396TLE7TLiPb2KXq,online,https://www.linkedin.com/jobs/view/staff-softw...,"Staff Software Engineer, Startup Products",Stripe,"New York, United States",2026-08-22T00:00:00,Mid-Senior level,Full-time,Engineering and Information Technology,"Software Development, Financial Services, and ...",3,stripe.com,2026-08-25T09:38:48.398106+00:00,NaN


In [ ]:
project_id = os.getenv("GCP_PROJECT_ID")

pandas_gbq.to_gbq(
    df,
    destination_table="sales_motion_raw.jobs",
    project_id=project_id,
    if_exists="replace",
    location="EU"
)

100%|██████████| 1/1 [00:00<00:00, 11554.56it/s]
